# **Naive RAG**
The Naive RAG is the simplest technique in the RAG ecosystem, providing a straightforward approach to combining retrieved data with LLM models for efficient user responses.

Research Paper: [RAG](https://arxiv.org/pdf/2005.11401)

## **Initial Setup**

In [1]:
%pip install -q sentence-transformers torch faiss-cpu ragas

^C
Note: you may need to restart the kernel to use updated packages.


## **Indexing**

In [2]:
# load embedding model
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("AITeamVN/Vietnamese_Embedding_v2", device=device)
model.max_seq_length = 2048

# quick semantic similarity check
sentences_1 = ["Tri tue nhan tao la gi", "Loi ich cua giac ngu"]
sentences_2 = [
    "Tri tue nhan tao la cong nghe giup may moc suy nghi va hoc hoi nhu con nguoi.",
    "Giac ngu giup co the va nao bo nghi ngoi, hoi phuc nang luong va cai thien tri nho.",
]
query_embedding = model.encode(sentences_1)
doc_embeddings = model.encode(sentences_2)
similarity = query_embedding @ doc_embeddings.T
print(similarity)

class VietnameseEmbedding:
    def __init__(self, sentence_model):
        self.sentence_model = sentence_model

    def embed_documents(self, texts):
        return self.sentence_model.encode(texts).tolist()

    def embed_query(self, text):
        return self.sentence_model.encode(text).tolist()

embeddings = VietnameseEmbedding(model)
embedding_dim = len(embeddings.embed_query("kiem tra kich thuoc embedding"))
print(f"Embedding dimension: {embedding_dim}")

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
# load data
from langchain.document_loaders import CSVLoader
loader = CSVLoader("./context.csv")
documents = loader.load()

In [ ]:
# split documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

## **FAISS Vector Database**

In [ ]:
# create FAISS vectorstore
from langchain.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, embeddings)

In [ ]:
# optional: save FAISS index to local disk
vectorstore.save_local("faiss_index")

In [ ]:
# optional: load FAISS index from local disk
from langchain.vectorstores import FAISS
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
# vectorstore ready
vectorstore

## **FAISS (Optional Tips)**

In [ ]:
# optional: quick FAISS retrieval check
docs = vectorstore.similarity_search("when did ww1 end?", k=2)
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.page_content[:160]}")

## **Retriever**

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

## **RAG Chain**

In [ ]:
# load llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
# create document chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

template = """"
You are a helpful assistant that answers questions based on the provided context.
Use the provided context to answer the question.
Question: {input}
Context: {context}
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# Setup RAG pipeline
rag_chain = (
    {"context": retriever,  "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# response
response = rag_chain.invoke("when did ww1 end?")
response

'World War I ended on November 11, 1918.'

## **Preparing Data for Evaluation**

In [ ]:
# create dataset
question = ["when did ww1 end?"]
response = []
contexts = []

# Inference
for query in question:
  response.append(rag_chain.invoke(query))
  contexts.append([docs.page_content for docs in retriever.get_relevant_documents(query)])

# To dict
data = {
    "query": question,
    "response": response,
    "context": contexts,
}

In [ ]:
# create dataset
from datasets import Dataset
dataset = Dataset.from_dict(data)

In [ ]:
# create dataframe
import pandas as pd
df = pd.DataFrame(dataset)

In [ ]:
df

,query,response,context
0,when did ww1 end?,"World War I ended on November 11, 1918.",[context: ['World War I or the First World War...


In [ ]:
# Convert to dictionary
df_dict = df.to_dict(orient='records')

# Convert context to list
for record in df_dict:
    if not isinstance(record.get('context'), list):
        if record.get('context') is None:
            record['context'] = []
        else:
            record['context'] = [record['context']]

## **Evaluation with RAGAS**

Su dung RAGAS de cham diem chat luong RAG theo cac metric pho bien nhu faithfulness, answer relevancy, context precision va context recall.

In [ ]:
# prepare ragas dataset
from datasets import Dataset

ground_truth = ["World War I ended on 11 November 1918."] * len(question)

ragas_data = {
    "question": question,
    "answer": response,
    "contexts": contexts,
    "ground_truth": ground_truth,
}

ragas_dataset = Dataset.from_dict(ragas_data)
ragas_dataset

In [ ]:
# run ragas evaluation
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall

try:
    from ragas.metrics import answer_relevancy
except ImportError:
    from ragas.metrics import answer_relevance as answer_relevancy

ragas_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

In [ ]:
# show ragas scores
ragas_result.to_pandas()

You can view your dataset at: https://app.athina.ai/develop/80872384-24ac-4ad9-824d-74dc02cb7cca


,query,context,response,expected_response,display_name,failed,grade_reason,runtime,model,passed
0,when did ww1 end?,"[context: ['World War I or the First World War (28 July 1914 – 11 November 1918), often abbreviated as WWI, was one of the deadliest global conflicts in history. It was fought between two coalitions, the Allies and the Central Powers. Fighting occurred throughout Europe, the Middle East, Africa, the Pacific, and parts of Asia. An estimated 9 million soldiers were killed in combat, plus another 23 million wounded, while 5 million civilians died as a result of military action, hunger, and dise...","World War I ended on November 11, 1918.",None,Does Response Answer Query,False,"The response directly answers the user's query by providing the specific date on which World War I ended, which is November 11, 1918. This sufficiently covers all aspects of the user's query.",787,gpt-4o,1.0
